# Nexus-Steg: Training Notebook

**Robust Semantic-Texture Hybrid Steganography System**

Assumes dataset is already on Drive. Runs: sanity check, overfit test, full training, evaluation.

**Runtime:** GPU (A100 recommended). Go to Runtime > Change runtime type > A100.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Update Drive Repo → Clone to Local → Install

In [ ]:
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/DL_Project/Nexus-Steg"
PROJECT_DIR = "/content/Nexus-Steg"
REPO_URL = "https://github.com/Henildiyora/Nexus-Steg.git"
BRANCH = "improvement/mod_training_pipeline"

# --- Step 1: Update the Drive repo ---
if os.path.exists(DRIVE_DIR):
    print(">>> Pulling latest changes in Drive repo …")
    %cd "$DRIVE_DIR"
    !git pull origin $BRANCH
else:
    print(f"Drive repo not found at {DRIVE_DIR}, skipping pull.")

# --- Step 2: Fresh clone into /content/ (local disk = faster I/O) ---
if os.path.exists(PROJECT_DIR):
    print(f">>> Removing old local clone at {PROJECT_DIR} …")
    shutil.rmtree(PROJECT_DIR)

print(">>> Cloning repo into /content/ …")
!git clone -b $BRANCH $REPO_URL "$PROJECT_DIR"

# --- Step 3: Install ---
%cd "$PROJECT_DIR"
!pip install -e . -q
print("\nRepo ready at:", PROJECT_DIR)

## 3a. Download Cover Images (MS-COCO val2017 + test2017, ~7 GB)

In [ ]:
import os, glob

%cd "$PROJECT_DIR"

COVER_DIR = os.path.join(PROJECT_DIR, "datasets", "cover")
os.makedirs(COVER_DIR, exist_ok=True)
existing_cover = len(glob.glob(os.path.join(COVER_DIR, "*")))

if existing_cover >= 40_000:
    print(f"Cover images already present ({existing_cover} files) — skipping.")
else:
    print(">>> Downloading MS-COCO val2017 (~1 GB) …")
    !curl -L --progress-bar -o datasets/val2017.zip "http://images.cocodataset.org/zips/val2017.zip"
    !unzip -q -o datasets/val2017.zip -d datasets/
    !mv datasets/val2017/* "$COVER_DIR/"
    !rm -rf datasets/val2017 datasets/val2017.zip

    print(">>> Downloading MS-COCO test2017 (~6 GB) …")
    !curl -L --progress-bar -o datasets/test2017.zip "http://images.cocodataset.org/zips/test2017.zip"
    !unzip -q -o datasets/test2017.zip -d datasets/
    !mv datasets/test2017/* "$COVER_DIR/"
    !rm -rf datasets/test2017 datasets/test2017.zip

print(f"Cover images: {len(glob.glob(os.path.join(COVER_DIR, '*')))} files")

## 3b. Download Secret Images (SpaceNet 2 MUL-PanSharpen, 4 cities, ~8 GB)

In [ ]:
import os, glob

%cd "$PROJECT_DIR"

SECRET_DIR = os.path.join(PROJECT_DIR, "datasets", "secret", "MUL-PanSharpen")
os.makedirs(SECRET_DIR, exist_ok=True)
existing_secret = len(glob.glob(os.path.join(SECRET_DIR, "*.tif")))

if existing_secret >= 100:
    print(f"Secret images already present ({existing_secret} files) — skipping.")
else:
    print(">>> Installing AWS CLI …")
    !pip install awscli -q

    S3_BUCKET = "s3://spacenet-dataset/spacenet/SN2_buildings"
    CITIES = ["AOI_2_Vegas", "AOI_3_Paris", "AOI_4_Shanghai", "AOI_5_Khartoum"]

    for city in CITIES:
        for s3_split in ["train", "test_public"]:
            s3_path = f"{S3_BUCKET}/{s3_split}/{city}/PS-MS/"
            print(f">>> Downloading {city} ({s3_split}) …")
            !aws s3 cp "$s3_path" "$SECRET_DIR/" --recursive --no-sign-request --quiet

print(f"Secret images: {len(glob.glob(os.path.join(SECRET_DIR, '*.tif')))} files")

## 4. Verify Dataset

In [ ]:
import os, glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

%cd "$PROJECT_DIR"

cover_files = sorted(glob.glob("datasets/cover/*.[jp][pn][g]*"))
secret_files = sorted(glob.glob("datasets/secret/MUL-PanSharpen/*.tif"))

print(f"Cover images:  {len(cover_files)}")
print(f"Secret images: {len(secret_files)}")
print(f"Training pairs (min): {min(len(cover_files), len(secret_files))}")

assert len(cover_files) > 0, "No cover images found!"
assert len(secret_files) > 0, "No secret images found!"

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Dataset Samples (top: covers, bottom: secrets)", fontsize=14)

for i in range(4):
    img = Image.open(cover_files[i * len(cover_files) // 4]).convert("RGB")
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Cover {i+1}")
    axes[0, i].axis("off")

import tifffile as tiff
for i in range(4):
    raw = tiff.imread(secret_files[i * len(secret_files) // 4])
    if raw.dtype == np.uint16:
        p2, p98 = np.percentile(raw, (2, 98))
        raw = np.clip(raw, p2, p98)
        raw = ((raw - p2) / max(p98 - p2, 1e-6) * 255).astype(np.uint8)
    if len(raw.shape) == 3:
        if raw.shape[0] < raw.shape[2]:
            raw = raw.transpose(1, 2, 0)
        raw = raw[:, :, :3]
    axes[1, i].imshow(raw)
    axes[1, i].set_title(f"Secret {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 5. Sanity Check

| Loss | Expected | If Wrong |
|------|----------|----------|
| `l_inv` | 0.01 - 0.10 | Encoder init problem |
| `l_rec` | 0.30 - 0.70 | Reveal network problem |
| `l_disc` | ~0.693 | Discriminator init problem |

In [ ]:
%cd "$PROJECT_DIR"
!python main.py --sanity --num_workers 2

## 6. Overfit One Batch

- **PASS** (loss < 0.01): Model capacity is sufficient.
- **WARN** (loss 0.01-0.10): Probably fine, monitor during training.
- **FAIL** (loss > 0.10): Architecture or LR problem.

In [ ]:
%cd "$PROJECT_DIR"
!python main.py --overfit_one_batch --num_workers 2

## 7. Train

| Phase | Epochs | What Happens |
|-------|--------|-------------|
| 1 | 0-29 | Pure hiding/recovery. No noise, no adversarial. |
| 2 | 30-59 | Noise layer ON, adversarial ON (mild). |
| 3 | 60-99 | Full adversarial pressure. Metrics should plateau. |

Early stopping after 15 epochs without improvement. Best checkpoint saved to `checkpoints/nexus_best.pth`.

In [ ]:
%cd "$PROJECT_DIR"
!python main.py --epochs 100 --batch_size 64 --checkpoint_every 10 --patience 15 --num_workers 2

## 8. Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%cd "$PROJECT_DIR"
df = pd.read_csv("results/training_log.csv")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Nexus-Steg Training Curves", fontsize=16)

ax = axes[0, 0]
ax.plot(df["epoch"], df["val_psnr_stego"], label="PSNR(stego)", linewidth=2)
ax.plot(df["epoch"], df["val_psnr_secret"], label="PSNR(secret)", linewidth=2)
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5, label="Phase 2")
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5, label="Phase 3")
ax.set_xlabel("Epoch"); ax.set_ylabel("PSNR (dB)")
ax.set_title("Validation PSNR"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(df["epoch"], df["val_ssim_stego"], label="SSIM(stego)", linewidth=2)
ax.plot(df["epoch"], df["val_ssim_secret"], label="SSIM(secret)", linewidth=2)
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("SSIM")
ax.set_title("Validation SSIM"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(df["epoch"], df["l_inv"], label="l_inv", linewidth=1.5)
ax.plot(df["epoch"], df["l_rec"], label="l_rec", linewidth=1.5)
ax.plot(df["epoch"], df["l_disc"], label="l_disc", linewidth=1.5)
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training Losses"); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(df["epoch"], df["lr"], linewidth=2, color="tab:orange")
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate")
ax.set_title("Cosine Annealing LR"); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()

## 9. Evaluate

Runs 8 attack tests: Basic Recovery, JPEG-90, JPEG-50, Blur, Noise, Screenshot, Social Media, Steganalysis.

In [ ]:
%cd "$PROJECT_DIR"
!python evaluate.py --checkpoint checkpoints/nexus_best.pth

## 10. Display Visual Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

%cd "$PROJECT_DIR"

epoch_imgs = []
for target in [0, 29, 59, 99]:
    path = f"results/epoch_{target}.png"
    if os.path.exists(path):
        epoch_imgs.append((target, path))

if not epoch_imgs:
    all_epochs = sorted(glob.glob("results/epoch_*.png"))
    for p in all_epochs[:4]:
        e = int(os.path.basename(p).split("_")[1].split(".")[0])
        epoch_imgs.append((e, p))

if epoch_imgs:
    fig, axes = plt.subplots(len(epoch_imgs), 1, figsize=(20, 5 * len(epoch_imgs)))
    if len(epoch_imgs) == 1:
        axes = [axes]
    for ax, (e, path) in zip(axes, epoch_imgs):
        ax.imshow(mpimg.imread(path))
        ax.set_title(f"Epoch {e}  [cover | secret | stego | revealed]", fontsize=13)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No epoch images found yet.")

print("\n--- Evaluation Attack Results ---\n")
eval_imgs = sorted(glob.glob("results/evaluation/*.png"))
if eval_imgs:
    fig, axes = plt.subplots(len(eval_imgs), 1, figsize=(20, 4 * len(eval_imgs)))
    if len(eval_imgs) == 1:
        axes = [axes]
    for ax, path in zip(axes, eval_imgs):
        ax.imshow(mpimg.imread(path))
        ax.set_title(os.path.splitext(os.path.basename(path))[0].replace("_", " ").title(), fontsize=12)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation images found. Run Cell 8 first.")

## 11. Evaluation Report

In [ ]:
import os
%cd "$PROJECT_DIR"

report_path = "results/evaluation/report.txt"
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())
else:
    print("No evaluation report found. Run Cell 8 first.")